In [1]:
import sys, os 
import geopandas as gpd 
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
data_path = os.path.join(os.getcwd(), '..', 'data', 'geojson', 'polygon_cleaned.geojson')
gdf = gpd.read_file(data_path)

In [2]:
gdf

,CODIGO,OBSERV,INSUMO,APOYO,ASIGNACION,Asignado,OAM,area,url_type,download_url,poly_id,image_uid,geometry
0,231,clean pasture,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,34429.374,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,0,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.97056 10.61903, -72.97054 ..."
1,21,seasonal crops,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,6784.769,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.96914 10.61884, -72.96915 ..."
2,112,urban not continuous,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,12227.436,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,2,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.96928 10.61948, -72.96931 ..."
3,231,clean pasture,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,19385.187,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,3,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.97264 10.61976, -72.97273 ..."
4,313,fragmented forest,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,5013.914,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,4,img_0000_c4a77f3e,"MULTIPOLYGON (((-72.96917 10.62053, -72.9691 1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1948,231,clean pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,1528.937,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1948,img_0029_37df798b,"MULTIPOLYGON (((-75.123 3.80901, -75.12297 3.8..."
1949,333,Bare areas,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,0.026,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1949,img_0029_37df798b,"MULTIPOLYGON (((-75.12391 3.81075, -75.1239 3...."
1950,232,wooded pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,10536.989,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1950,img_0029_37df798b,"MULTIPOLYGON (((-75.12278 3.8104, -75.12282 3...."
1951,231,clean pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,5139.198,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,1951,img_0029_37df798b,"MULTIPOLYGON (((-75.12367 3.81079, -75.12357 3..."


### Build the COG URL based on the image_uid 

We need to build the COG URL so we can query the image stats for each polygon.

In [3]:
base_url = os.path.join(os.getcwd(), '..', 'data', 'images', 'cog')
row = gdf.iloc[0]
cog_url = f"{base_url}/{row['image_uid']}.tif"
geom = [row.geometry.__geo_interface__]
cog_url

'/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/notebooks/../data/images/cog/img_0000_c4a77f3e.tif'

## Reprojection and computation 

First, we handle the reprojection issue. The drone images are in a local projected coordinate system of WGS84, but the geometry is in degrees following EPSG:4326, so we need to project the geometry into the CRS of the raster (it is easier this way than the other way around). After that, we take advantage of the tiles we constructed earlier with COG - we only fetch the part of the image for that polygon. Once we have that, we have the band data. We have 3 bands in the image.

We observed mainly two main artifacts in the drone images: 
- First: there are no-data pixels in the drone images, especially near the boundaries and sometimes randomly in the middle, mostly because of how the drone image was processed and collected - maybe there were no overlapping pixels.
- We also observed that the pixel values are 0 in random areas, possibly seeming like faulty pixels, or we can say invalid pixels because the values were not even in the 0 to 255 range. 

Hence, to handle them, we first used a function from numpy's masked array that helps us identify the invalid pixels, and then we filtered the nodata values in the raster, which helped us remove those no-data pixels. We will study in depth whether they would have any impact on the mean computations!

In [4]:
import rasterio 
from rasterio.mask import mask
import numpy as np
from skimage import color
from scipy.stats import circmean, circstd

with rasterio.open(cog_url) as src:
    print("Raster CRS:", src.crs)
    print("Raster bounds:", src.bounds)
    
    print("\nOriginal geometry CRS:", gdf.crs)
    print("Original geometry bounds:", row.geometry.bounds)
    
    geom_gdf = gpd.GeoDataFrame([row], geometry='geometry', crs=gdf.crs)
    geom_reprojected = geom_gdf.to_crs(src.crs)
    geom_transformed = [geom_reprojected.geometry.iloc[0].__geo_interface__]
    
    print("\nTransformed bounds:", geom_reprojected.geometry.iloc[0].bounds)
    
    masked_data, _ = mask(src, geom_transformed, crop=True, all_touched=False)

    r_band = masked_data[0]
    g_band = masked_data[1]
    b_band = masked_data[2]
    
    is_masked = np.ma.isMaskedArray(r_band)
    
    if is_masked:
        valid_mask = ~(r_band.mask | g_band.mask | b_band.mask)
    else:
        nodata = src.nodata if src.nodata is not None else -9999
        valid_mask = (r_band != nodata) & (g_band != nodata) & (b_band != nodata)
    
    r_valid = r_band[valid_mask]
    g_valid = g_band[valid_mask]
    b_valid = b_band[valid_mask]
    
    stats = {}
    
    if r_valid.size > 0:
        stats['r_mean'] = float(np.mean(r_valid))
        stats['r_median'] = float(np.median(r_valid))
        stats['r_std'] = float(np.std(r_valid))
        stats['r_var'] = float(np.var(r_valid))
        
        stats['g_mean'] = float(np.mean(g_valid))
        stats['g_median'] = float(np.median(g_valid))
        stats['g_std'] = float(np.std(g_valid))
        stats['g_var'] = float(np.var(g_valid))
        
        stats['b_mean'] = float(np.mean(b_valid))
        stats['b_median'] = float(np.median(b_valid))
        stats['b_std'] = float(np.std(b_valid))
        stats['b_var'] = float(np.var(b_valid))
        
        r_g = np.divide(r_valid, g_valid, where=g_valid != 0, out=np.full_like(r_valid, np.nan, dtype=float))
        r_b = np.divide(r_valid, b_valid, where=b_valid != 0, out=np.full_like(r_valid, np.nan, dtype=float))
        g_b = np.divide(g_valid, b_valid, where=b_valid != 0, out=np.full_like(g_valid, np.nan, dtype=float))
        
        stats['r_g_mean'] = float(np.nanmean(r_g))
        stats['r_g_std'] = float(np.nanstd(r_g))
        stats['r_b_mean'] = float(np.nanmean(r_b))
        stats['r_b_std'] = float(np.nanstd(r_b))
        stats['g_b_mean'] = float(np.nanmean(g_b))
        stats['g_b_std'] = float(np.nanstd(g_b))
        
        r_minus_g = r_valid - g_valid
        g_minus_b = g_valid - b_valid
        
        stats['r_minus_g_mean'] = float(np.mean(r_minus_g))
        stats['r_minus_g_std'] = float(np.std(r_minus_g))
        stats['g_minus_b_mean'] = float(np.mean(g_minus_b))
        stats['g_minus_b_std'] = float(np.std(g_minus_b))
        
        rgb_stack = np.stack([r_band[valid_mask], g_band[valid_mask], b_band[valid_mask]], axis=-1)
        rgb_norm = rgb_stack / 255.0 if rgb_stack.max() > 1 else rgb_stack
        
        hsv = color.rgb2hsv(rgb_norm.reshape(1, -1, 3)).reshape(-1, 3)
        stats['hsv_s_mean'] = float(np.mean(hsv[:, 1]))
        stats['hsv_v_mean'] = float(np.mean(hsv[:, 2]))
        
        hue_rad = hsv[:, 0] * 2 * np.pi
        stats['hsv_h_mean'] = float(circmean(hue_rad))
        stats['hsv_h_std'] = float(circstd(hue_rad))
        
        lab = color.rgb2lab(rgb_norm.reshape(1, -1, 3)).reshape(-1, 3)
        stats['lab_l_mean'] = float(np.mean(lab[:, 0]))
        stats['lab_a_mean'] = float(np.mean(lab[:, 1]))
        stats['lab_b_mean'] = float(np.mean(lab[:, 2]))
        
        denom = g_valid + r_valid - b_valid
        vari = np.divide(g_valid - r_valid, denom, where=denom != 0, out=np.full_like(g_valid, np.nan, dtype=float))
        
        stats['vari_mean'] = float(np.nanmean(vari))
        stats['vari_std'] = float(np.nanstd(vari))
    else:
        stats = {k: np.nan for k in [
            'r_mean', 'r_median', 'r_std', 'r_var',
            'g_mean', 'g_median', 'g_std', 'g_var',
            'b_mean', 'b_median', 'b_std', 'b_var',
            'r_g_mean', 'r_g_std', 'r_b_mean', 'r_b_std', 'g_b_mean', 'g_b_std',
            'r_minus_g_mean', 'r_minus_g_std', 'g_minus_b_mean', 'g_minus_b_std',
            'hsv_s_mean', 'hsv_v_mean', 'hsv_h_mean', 'hsv_h_std',
            'lab_l_mean', 'lab_a_mean', 'lab_b_mean',
            'vari_mean', 'vari_std'
        ]}
    
    print(f"Total features: {len(stats)}")
    print(stats)

Raster CRS: EPSG:32618
Raster bounds: BoundingBox(left=721782.2054845318, bottom=1174560.63571348, right=722310.3216997313, top=1174796.2209395552)

Original geometry CRS: EPSG:4326
Original geometry bounds: (-72.9714814422654, 10.618840012053344, -72.96914035652021, 10.620975138959965)

Transformed bounds: (721925.7615761297, 1174560.7526012938, 722182.7255575805, 1174796.086679114)
Total features: 31
{'r_mean': 67.11867219203522, 'r_median': 83.0, 'r_std': 63.05077827901257, 'r_var': 3975.4006415892036, 'g_mean': 72.36945632182152, 'g_median': 101.0, 'g_std': 65.61500049739803, 'g_var': 4305.328290273544, 'b_mean': 50.0079449993589, 'b_median': 55.0, 'b_std': 48.51453627072589, 'b_var': 2353.660229563578, 'r_g_mean': 0.9220828022867901, 'r_g_std': 0.1619652343767181, 'r_b_mean': 1.4110471493459107, 'r_b_std': 0.6423001382293005, 'g_b_mean': 1.6244731687820582, 'g_b_std': 1.351284937226633, 'r_minus_g_mean': 95.77257832063441, 'r_minus_g_std': 114.09794140042416, 'g_minus_b_mean': 22.

### Build the features 

Here we try to extract different types of statistics from the image in our polygon. We have already calculated the area of the polygon that can be used as a feature, and here we are using mean, std, var in each band as well as some band indices, green index, and color profile!

In [5]:
from src.stats import compute_raster_stats

In [ ]:
gdf_with_stats = compute_raster_stats(gdf, base_url='/home/krschap/Jupyter/ubs/ml/mlproject1/landcoverclassification/data/images/cog')

test


Processing polygons:   0%|          | 0/1953 [00:00<?, ?it/s]

In [14]:
gdf_with_stats

,CODIGO,OBSERV,INSUMO,APOYO,ASIGNACION,Asignado,OAM,area,url_type,download_url,...,g_minus_b_std,hsv_s_mean,hsv_v_mean,hsv_h_mean,hsv_h_std,lab_l_mean,lab_a_mean,lab_b_mean,vari_mean,vari_std
0,231,clean pasture,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,34429.374,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,21,seasonal crops,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,6784.769,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,112,urban not continuous,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,12227.436,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,231,clean pasture,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,19385.187,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,313,fragmented forest,Villanueva-orthophoto,None,Sofia,None,https://map.openaerialmap.org/#/-72.9698181152...,5013.914,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1948,231,clean pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,1528.937,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1949,333,Bare areas,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,0.026,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1950,232,wooded pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,10536.989,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1951,231,clean pasture,668c76331684770001c33863,None,None,Juan,https://map.openaerialmap.org/#/-75.1227307319...,5139.198,oam,https://oin-hotosm-temp.s3.us-east-1.amazonaws...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
gdf_with_stats.to_file(os.path.join(os.getcwd(), '..', 'data', 'geojson', 'polygon_cleaned_with_stats.geojson'))